In [1]:
import subprocess, sys, os

print(" Checking GPU...")
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
print(f"   GPU: {r.stdout.strip()}")

import torch
assert torch.cuda.is_available(), " No GPU! Switch to T4 in Runtime → Change runtime type."
print(f"    {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB VRAM\n")


 Checking GPU...
   GPU: Tesla T4, 15360 MiB
    Tesla T4 | 14.6 GB VRAM



In [3]:
# ═════════════════════════════════════════════════════════════════════════════
#  INSTALLATION PACKAGES — VERSION COMPATIBLE (transformers==4.51.3)
# ═════════════════════════════════════════════════════════════════════════════
import sys, subprocess, pkg_resources

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

print(" [1/8] Core ML deps...")
pip("bitsandbytes>=0.43.0", "peft>=0.10.0", "trl>=0.8.6", "accelerate>=0.28.0", "xformers")

print(" [2/8] Transformers PINNÉ + Unsloth...")
#  CRITIQUE : Pin transformers AVANT unsloth pour éviter l'upgrade automatique
pip("transformers==4.51.3")
pip("datasets>=2.19.0", "huggingface_hub>=0.22.0", "tokenizers>=0.19.0")
pip("unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git")

print("[3/8] FAISS + retrieval...")
pip("faiss-cpu")
pip("sentence-transformers>=3.0.0,<4.0.0", "rank-bm25", "rouge-score", "scikit-learn")

print(" [4/8] Document parsers...")
pip("pymupdf", "pdfplumber", "python-docx", "python-pptx", "openpyxl",
    "ebooklib", "beautifulsoup4", "lxml", "chardet", "wikipedia-api", "requests")

print(" [5/8] Gradio...")
pip("gradio>=4.31.0")

print(" [6/8] RAGAS + LangChain...")
pip("ragas", "langchain-huggingface", "langchain")

print(" [7/8] NLTK data...")
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print(" [8/8]  FORCE FINAL: transformers==4.51.3 (override post-unsloth)...")
#  CRITIQUE : Re-forcer la version après toutes les installations
pip("transformers==4.51.3", "--force-reinstall", "-q")

print("\n Vérification...")
import transformers, sentence_transformers
print(f"   transformers: {transformers.__version__}")
print(f"   sentence-transformers: {sentence_transformers.__version__}")

try:
    import faiss
    print(f"   faiss: {faiss.__version__}")
except:
    pass

print("\n PACKAGES INSTALLÉS !")
print(" MAINTENANT : Runtime → Restart session (OBLIGATOIRE)")

 [1/8] Core ML deps...
 [2/8] Transformers PINNÉ + Unsloth...
[3/8] FAISS + retrieval...
 [4/8] Document parsers...
 [5/8] Gradio...
 [6/8] RAGAS + LangChain...
 [7/8] NLTK data...
 [8/8]  FORCE FINAL: transformers==4.51.3 (override post-unsloth)...

 Vérification...


   transformers: 4.51.3
   sentence-transformers: 3.4.1
   faiss: 1.14.2

 PACKAGES INSTALLÉS !
 MAINTENANT : Runtime → Restart session (OBLIGATOIRE)


In [2]:
# STEP 2 ── IMPORTS & CONFIG
# ══════════════════════════════════════════════════════════════════════════════
import gc, re, json, time, warnings, tempfile
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

import numpy as np
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {DEVICE} | {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB\n")

@dataclass
class Config:
    # ── Models ────────────────────────────────────────────────────────────────
    LLM_MODEL:      str = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
    EMBED_MODEL:    str = "BAAI/bge-large-en-v1.5"
    RERANKER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # ── RAG retrieval ─────────────────────────────────────────────────────────
    CHUNK_SIZE:     int   = 350
    CHUNK_OVERLAP:  int   = 50
    TOP_K_RETRIEVE: int   = 20
    TOP_K_RERANK:   int   = 6
    BM25_WEIGHT:    float = 0.35
    DENSE_WEIGHT:   float = 0.65

    # ── HuggingFace datasets (lightweight — max 150 rows each) ────────────────
    HF_DATASETS: List[Dict] = field(default_factory=lambda: [
        {"name": "cnn_dailymail", "config": "3.0.0", "split": "train[:150]",
         "text_col": "article",  "sum_col": "highlights", "domain": "news"},
        {"name": "EdinburghNLP/xsum", "config": None, "split": "train[:100]",
         "text_col": "document", "sum_col": "summary",    "domain": "news"},
        {"name": "billsum", "config": None, "split": "train[:80]",
         "text_col": "text",     "sum_col": "summary",    "domain": "legal"},
        {"name": "knkarthick/dialogsum", "config": None, "split": "train[:80]",
         "text_col": "dialogue", "sum_col": "summary",    "domain": "dialogue"},
    ])

    # ── Wikipedia articles for aviation/technical domain ──────────────────────
    WIKIPEDIA_ARTICLES: List[str] = field(default_factory=lambda: [
        "CFM56", "Turbofan", "Jet engine", "Gas turbine",
        "Compressor stall", "Foreign object damage", "Turbine blade",
        "Aircraft maintenance", "Borescope", "Airworthiness directive",
        "Non-destructive testing", "Predictive maintenance",
        "Remaining useful life", "Health and usage monitoring systems",
        "Federal Aviation Administration", "European Union Aviation Safety Agency",
        "Metal fatigue", "Creep (deformation)", "Turbine blade cooling",
        "Artificial intelligence", "Transformer (machine learning)",
        "Retrieval-augmented generation", "Large language model",
    ])

    # ── Generation defaults ────────────────────────────────────────────────────
    MAX_INPUT_WORDS:    int   = 4000
    DEFAULT_MAX_TOKENS: int   = 512
    DEFAULT_TEMP:       float = 0.2

CFG = Config()
print(f" Config loaded — LLM: {CFG.LLM_MODEL}\n")


 Device: cuda | Tesla T4 | 14.6 GB

 Config loaded — LLM: unsloth/mistral-7b-instruct-v0.3-bnb-4bit



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ══════════════════════════════════════════════════════════════════════
# STEP 3 ── MULTI-FORMAT DOCUMENT INGESTION ENGINE (CLEAN)
# ══════════════════════════════════════════════════════════════════════

import os, re, json, tempfile
from pathlib import Path
from typing import List, Dict

import fitz
import pdfplumber
import docx as _docx
from pptx import Presentation as _PRS
import openpyxl
from bs4 import BeautifulSoup
import chardet
import pandas as pd

try:
    import ebooklib
    from ebooklib import epub
    EPUB_OK = True
except Exception:
    EPUB_OK = False


# ─────────────────────────────────────────────────────────────
# ENGINE
# ─────────────────────────────────────────────────────────────

class DocumentIngestionEngine:
    """Universal parser: PDF, DOCX, PPTX, XLSX, TXT, MD, HTML, CSV, JSON, EPUB"""

    SUPPORTED = {
        ".pdf": "PDF", ".docx": "Word", ".doc": "Word",
        ".pptx": "PowerPoint", ".xlsx": "Excel", ".xls": "Excel",
        ".txt": "Text", ".rtf": "RTF", ".md": "Markdown",
        ".html": "HTML", ".htm": "HTML",
        ".csv": "CSV", ".json": "JSON", ".epub": "eBook",
    }

    # ─────────────────────────────────────────────
    def load(self, file_path: str) -> Dict:
        path = Path(file_path)
        ext = path.suffix.lower()

        if ext not in self.SUPPORTED:
            raise ValueError(f"Unsupported format: {ext}")

        dispatch = {
            ".pdf": self._pdf,
            ".docx": self._docx,
            ".doc": self._docx,
            ".pptx": self._pptx,
            ".xlsx": self._xlsx,
            ".xls": self._xlsx,
            ".txt": self._text,
            ".rtf": self._text,
            ".md": self._text,
            ".html": self._html,
            ".htm": self._html,
            ".csv": self._csv,
            ".json": self._json,
            ".epub": self._epub,
        }

        text, extra = dispatch[ext](str(path))

        meta = {
            "filename": path.name,
            "format": self.SUPPORTED[ext],
            "words": len(text.split()),
            "chars": len(text),
        }
        meta.update(extra)

        return {"text": text.strip(), "metadata": meta}

    # ─────────────────────────────────────────────
    def load_bytes(self, data: bytes, filename: str) -> Dict:
        ext = Path(filename).suffix.lower()
        with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
            tmp.write(data)
            tmp_path = tmp.name

        try:
            res = self.load(tmp_path)
            res["metadata"]["filename"] = filename
            return res
        finally:
            os.unlink(tmp_path)

    # ─────────────────────────────────────────────
    def _pdf(self, p):
        pages = []
        try:
            doc = fitz.open(p)
            for page in doc:
                pages.append(page.get_text("text"))
            doc.close()
        except:
            pass

        text = "\n".join(pages)
        return text, {"pages": len(pages)}

    # ─────────────────────────────────────────────
    def _docx(self, p):
        doc = _docx.Document(p)
        text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
        return text, {"pages": len(doc.paragraphs)}

    # ─────────────────────────────────────────────
    def _pptx(self, p):
        prs = _PRS(p)
        slides = []
        for i, slide in enumerate(prs.slides):
            parts = []
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text:
                    parts.append(shape.text)
            slides.append("\n".join(parts))
        return "\n\n".join(slides), {"pages": len(prs.slides)}

    # ─────────────────────────────────────────────
    def _xlsx(self, p):
        wb = openpyxl.load_workbook(p, read_only=True)
        text = ""
        for name in wb.sheetnames:
            ws = wb[name]
            text += f"\nSheet: {name}\n"
            for row in ws.iter_rows(values_only=True):
                text += " | ".join([str(c) if c else "" for c in row]) + "\n"
        return text, {"pages": len(wb.sheetnames)}

    # ─────────────────────────────────────────────
    def _html(self, p):
        raw = open(p, "rb").read()
        enc = chardet.detect(raw)["encoding"] or "utf-8"
        soup = BeautifulSoup(raw.decode(enc, errors="ignore"), "lxml")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        return soup.get_text("\n", strip=True), {}

    # ─────────────────────────────────────────────
    def _text(self, p):
        raw = open(p, "rb").read()
        enc = chardet.detect(raw)["encoding"] or "utf-8"
        return raw.decode(enc, errors="ignore"), {}

    # ─────────────────────────────────────────────
    def _csv(self, p):
        df = pd.read_csv(p, nrows=500)
        return df.to_string(), {"rows": len(df)}

    # ─────────────────────────────────────────────
    def _json(self, p):
        with open(p) as f:
            data = json.load(f)
        return json.dumps(data, indent=2)[:50000], {}

    # ─────────────────────────────────────────────
    def _epub(self, p):
        if not EPUB_OK:
            return "EPUB not supported", {}
        book = epub.read_epub(p)
        text = ""
        for item in book.get_items():
            text += str(item)
        return text, {}


# ─────────────────────────────────────────────────────────────
# INIT ENGINE
# ─────────────────────────────────────────────────────────────
DOC_ENGINE = DocumentIngestionEngine()

print("✔ DocumentIngestionEngine ready")

✔ DocumentIngestionEngine ready


In [5]:
# ══════════════════════════════════════════════════════════════════════
# STEP 4 ── BUILD RAG KNOWLEDGE BASE
#   4a. HuggingFace datasets (summarization examples)
#   4b. Wikipedia articles (aviation + technical domain)
#   4c. Built-in seed examples (always present as anchors)
#   4d. Local PDFs (technical manuals + FAA regulations)
# ══════════════════════════════════════════════════════════════════════
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import shutil, wikipediaapi

def smart_chunk(text: str, max_words: int = CFG.CHUNK_SIZE,
                overlap: int = CFG.CHUNK_OVERLAP) -> List[str]:
    sentences = sent_tokenize(text)
    chunks, cur, cur_w = [], [], 0
    for sent in sentences:
        sw = len(sent.split())
        if cur_w + sw > max_words and cur:
            chunks.append(" ".join(cur))
            ovl, ow = [], 0
            for s in reversed(cur):
                w = len(s.split())
                if ow + w <= overlap: ovl.insert(0, s); ow += w
                else: break
            cur, cur_w = ovl, ow
        cur.append(sent); cur_w += sw
    if cur: chunks.append(" ".join(cur))
    return chunks

def clean_text(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── 4a: HuggingFace datasets ─────────────────────────────────────────────────
print("\n [4a] Loading HuggingFace summarization datasets...")
rag_chunks: List[Dict] = []
hf_cache   = Path("/root/.cache/huggingface/datasets")

for ds_cfg in CFG.HF_DATASETS:
    name = ds_cfg["name"]
    try:
        print(f"    {name} ...", end=" ", flush=True)
        kwargs = {"split": ds_cfg["split"]}
        if ds_cfg["config"]: kwargs["name"] = ds_cfg["config"]
        ds     = load_dataset(name, **kwargs)
        avail  = ds.column_names
        tcol   = next((c for c in [ds_cfg["text_col"]] + ["text","article","document","content"]
                       if c in avail), avail[0])
        scol   = next((c for c in [ds_cfg["sum_col"]] + ["summary","highlights","abstract","headline"]
                       if c in avail), avail[1])
        before = len(rag_chunks)
        for row in ds:
            txt  = str(row.get(tcol, "")).strip()
            summ = str(row.get(scol, "")).strip()
            if len(txt) < 80 or len(summ) < 15: continue
            for chunk in smart_chunk(txt):
                if len(chunk.split()) >= 30:
                    rag_chunks.append({"text": chunk, "summary": summ,
                                       "domain": ds_cfg["domain"], "source": name,
                                       "chunk_id": f"{name}_{len(rag_chunks)}"})
        added = len(rag_chunks) - before
        print(f" +{added:,} chunks")
        # free disk
        ds_short = name.split("/")[-1]
        for p in hf_cache.glob(f"{ds_short}*"):
            shutil.rmtree(p, ignore_errors=True)
        gc.collect()
    except Exception as e:
        print(f" SKIP ({str(e)[:80]})")

# ── 4b: Wikipedia articles ────────────────────────────────────────────────────
print(f"\n [4b] Downloading {len(CFG.WIKIPEDIA_ARTICLES)} Wikipedia articles...")
wiki = wikipediaapi.Wikipedia(user_agent="UltimateRAGBot/1.0 (research)", language="en")
wiki_ok = 0
for title in CFG.WIKIPEDIA_ARTICLES:
    try:
        page = wiki.page(title)
        if page.exists() and len(page.text) > 500:
            text = clean_text(page.text)
            for chunk in smart_chunk(text):
                if len(chunk.split()) >= 30:
                    rag_chunks.append({"text": chunk, "summary": page.summary[:300],
                                       "domain": "technical", "source": f"Wikipedia:{title}",
                                       "chunk_id": f"wiki_{len(rag_chunks)}"})
            wiki_ok += 1
        time.sleep(0.3)
    except Exception:
        pass
print(f"    {wiki_ok}/{len(CFG.WIKIPEDIA_ARTICLES)} articles -> {len(rag_chunks):,} total chunks")

# ── 4c: Built-in seed examples ───────────────────────────────────────────────
print("\n [4c] Adding built-in seed examples...")
SEED_EXAMPLES = [
    ("The transformer architecture uses multi-head self-attention. "
     "Attention(Q,K,V)=softmax(QK^T/sqrt(d_k))V. BERT-base: 12L/768H/110M params.",
     "Transformer uses scaled dot-product attention; BERT-base has 12 layers, 768 dims, 110M params.", "technical"),
    ("Turbofan engines use bypass air for extra thrust and fuel efficiency. "
     "The CFM56 is the world's best-selling commercial jet engine, used on Boeing 737 and Airbus A320.",
     "CFM56 is the best-selling turbofan, used on 737/A320, using bypass air for efficiency.", "technical"),
    ("Airworthiness Directives (ADs) are legally enforceable FAA regulations requiring mandatory "
     "maintenance actions. Non-compliance can ground aircraft and revoke airworthiness certificates.",
     "ADs are FAA mandatory maintenance orders; non-compliance grounds aircraft.", "legal"),
    ("Q3 revenue was $12.4B (+18.3% YoY). Cloud $5.1B (+34%). EPS guidance $8.40-8.60. "
     "$2B buyback approved. Headcount +4,200.",
     "Q3: revenue $12.4B +18%, cloud $5.1B +34%, EPS $8.40-8.60, $2B buyback.", "financial"),
    ("Alice: Q3 report done. Bob: Include revenue breakdown? "
     "Alice: Cloud +34%, enterprise +22%. Bob: Flag EU review high priority.",
     "Alice completed Q3 report (cloud +34%, enterprise +22%); Bob flagged EU review as high priority.", "dialogue"),
    ("Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. "
     "A query is encoded, similar documents are retrieved from a vector store (FAISS), and the LLM "
     "uses them as context to produce grounded answers.",
     "RAG retrieves relevant documents via vector search (FAISS) and feeds them as context to the LLM.", "technical"),
    ("Compressor stall occurs when airflow through the compressor separates from blade surfaces, "
     "causing a sudden disruption. Symptoms: banging sounds, fluctuating thrust, possible engine shutdown.",
     "Compressor stall is blade airflow separation causing thrust disruption and engine noise.", "technical"),
]
for txt, summ, dom in SEED_EXAMPLES:
    rag_chunks.append({"text": txt, "summary": summ, "domain": dom,
                       "source": "builtin", "chunk_id": f"builtin_{len(rag_chunks)}"})

# ── 4d: Local PDFs ────────────────────────────────────────────────────────────
print("\n [4d] Loading local PDFs...")

PDF_BASE_DIR = "/content/drive/MyDrive/industrial-ai-platform/NLP/Data"

PDF_FOLDERS = {
    "technical": "technical",
    "faa": "legal",
}

pdf_total, pdf_ok = 0, 0
for folder, domain in PDF_FOLDERS.items():
    folder_path = f"{PDF_BASE_DIR}/{folder}"
    if not os.path.exists(folder_path):
        print(f"    WARNING: folder {folder}/ not found, skipping")
        continue

    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]
    print(f"\n    {folder}/ ({len(pdf_files)} PDFs) -> domain '{domain}'")

    for pdf_name in pdf_files:
        pdf_path = f"{folder_path}/{pdf_name}"
        pdf_total += 1
        try:
            print(f"       {pdf_name[:50]}...", end=" ", flush=True)
            doc = DOC_ENGINE.load(pdf_path)
            text = clean_text(doc["text"])
            chunk_count = 0
            for chunk in smart_chunk(text):
                if len(chunk.split()) >= 30:
                    rag_chunks.append({
                        "text": chunk,
                        "summary": doc["metadata"].get("title", "")[:300],
                        "domain": domain,
                        "source": f"PDF:{folder}/{pdf_name}",
                        "chunk_id": f"pdf_{len(rag_chunks)}"
                    })
                    chunk_count += 1
            pdf_ok += 1
            print(f"OK ({chunk_count} chunks)")
        except Exception as e:
            print(f"FAIL ({str(e)[:60]})")

print(f"\n    {pdf_ok}/{pdf_total} PDFs loaded -> {len(rag_chunks):,} total chunks")

# ── Domain stats ──────────────────────────────────────────────────────────────
domain_counts: Dict[str, int] = defaultdict(int)
for c in rag_chunks: domain_counts[c["domain"]] += 1
print(f"\n RAG Knowledge Base: {len(rag_chunks):,} chunks")
for dom, cnt in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"   {dom:20s}: {cnt:,}")



 [4a] Loading HuggingFace summarization datasets...
    cnn_dailymail ...  +353 chunks
    EdinburghNLP/xsum ...  +164 chunks
    billsum ...  +395 chunks
    knkarthick/dialogsum ...  +80 chunks

 [4b] Downloading 23 Wikipedia articles...
    22/23 articles -> 1,343 total chunks

 [4c] Adding built-in seed examples...

 [4d] Loading local PDFs...

    technical/ (7 PDFs) -> domain 'technical'
       ilide.info-72-ge-01-rev-3-turbofan-engine-troubles... OK (44 chunks)
       ilide.info-turbofan-engine-pr_a9b812811bcbac88c21a... OK (7 chunks)
       ilide.info-8f6eb6-pr_e5bc190a0bddddc33fac3c9f1930b... OK (66 chunks)
       e402102.pdf... OK (9 chunks)
       MMEL_DC-10r-25apt121.pdf... OK (156 chunks)
       T.O.1C-10KA-1-1-FlightManualPerformanceData-KC-10A... OK (430 chunks)
       Airplane turbofan engine operation and malfunction... OK (43 chunks)

    faa/ (4 PDFs) -> domain 'legal'
       faa_ac120_engine_interval.pdf... OK (11 chunks)
       faa_ac20_turbofan_icing.pdf... OK (7

In [ ]:
!pip uninstall transformers sentence-transformers -y -q
!pip install --upgrade "sentence-transformers>=3.0.0" "transformers>=4.40.0" -q

#  Vérification des versions installées
import transformers, sentence_transformers
print(f" transformers: {transformers.__version__}")
print(f" sentence-transformers: {sentence_transformers.__version__}")

In [6]:
#  Vérification des versions installées
import transformers, sentence_transformers
print(f" transformers: {transformers.__version__}")
print(f" sentence-transformers: {sentence_transformers.__version__}")

 transformers: 4.51.3
 sentence-transformers: 3.4.1


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 ── EMBEDDINGS + FAISS + BM25 (HYBRID RETRIEVER)
#   1. Tokenisation BM25 avec NLTK (stemming + stop words)
#   2. Seuil FAISS FlatIP étendu à 10 000 (au lieu de 200)
#   3. Sauvegarde/chargement persistant de l'index FAISS
#   4. Gestion OOM GPU avec réduction dynamique du batch
#   5. nprobe configurable via Config
# ══════════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss

# ── NLTK setup for BM25 tokenization ──────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

_stemmer = PorterStemmer()
_stop_words = set(stopwords.words('english'))

def tokenize_bm25(text: str) -> List[str]:
    """Tokenisation propre pour BM25 : lower, alpha-only, no stop words, stem."""
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in _stop_words and len(t) > 2]
    tokens = [_stemmer.stem(t) for t in tokens]
    return tokens

# ── FAISS persistence paths ───────────────────────────────────────────────────
FAISS_SAVE_DIR = Path("/content/drive/MyDrive/industrial-ai-platform/NLP/index_cache")
FAISS_SAVE_DIR.mkdir(parents=True, exist_ok=True)
FAISS_INDEX_PATH = FAISS_SAVE_DIR / "faiss.index"
FAISS_META_PATH  = FAISS_SAVE_DIR / "faiss_meta.json"

# ── Load or build embeddings ──────────────────────────────────────────────────
print(f"\n Loading embedding model: {CFG.EMBED_MODEL}...")
EMBED_MODEL = SentenceTransformer(CFG.EMBED_MODEL, device=DEVICE)
EMBED_DIM   = EMBED_MODEL.get_sentence_embedding_dimension()
print(f"   dim = {EMBED_DIM}")

print(f" Loading reranker: {CFG.RERANKER_MODEL}...")
RERANKER = CrossEncoder(CFG.RERANKER_MODEL, device=DEVICE)

# Check for cached index
texts = [c["text"] for c in rag_chunks]
N = len(texts)

if FAISS_INDEX_PATH.exists() and FAISS_META_PATH.exists():
    try:
        print(f"\n Found cached FAISS index, loading...")
        FAISS_INDEX = faiss.read_index(str(FAISS_INDEX_PATH))
        with open(FAISS_META_PATH) as f:
            meta = json.load(f)
        if meta.get("chunk_count") == N and meta.get("embed_model") == CFG.EMBED_MODEL:
            EMBEDDINGS = None  # Not needed if index loaded from disk
            print(f"   Loaded {FAISS_INDEX.ntotal:,} vectors from cache")
            index_loaded = True
        else:
            print(f"   Cache mismatch (chunks: {meta.get('chunk_count')} vs {N}), rebuilding...")
            index_loaded = False
    except Exception as e:
        print(f"   Cache load failed ({str(e)[:60]}), rebuilding...")
        index_loaded = False
else:
    print(f"\n No cache found, building index from scratch...")
    index_loaded = False

if not index_loaded:
    print(f"\n Encoding {N:,} chunks...")
    BATCH    = 128
    all_embs = []

    for i in range(0, len(texts), BATCH):
        batch_texts = texts[i:i+BATCH]
        try:
            embs = EMBED_MODEL.encode(batch_texts, normalize_embeddings=True,
                                       convert_to_numpy=True, show_progress_bar=False)
            all_embs.append(embs)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                BATCH = max(8, BATCH // 2)
                print(f"\n   WARNING: OOM detected, reducing batch to {BATCH}")
                torch.cuda.empty_cache()
                # Retry with smaller batch
                embs = EMBED_MODEL.encode(batch_texts, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False,
                                           batch_size=BATCH)
                all_embs.append(embs)
            else:
                raise

        pct = min(100, int((i + len(batch_texts)) / max(len(texts), 1) * 100))
        print(f"   {pct:3d}%...", end="\r")

    EMBEDDINGS = np.vstack(all_embs).astype("float32")
    print(f"\n   Embeddings shape: {EMBEDDINGS.shape}")

    # Build FAISS index
    if N < 10000:
        FAISS_INDEX = faiss.IndexFlatIP(EMBED_DIM)
        FAISS_INDEX.add(EMBEDDINGS)
        print(f"   FAISS FlatIP: {FAISS_INDEX.ntotal:,} vectors (exact search)")
    else:
        M = 64
        FAISS_INDEX = faiss.IndexHNSWFlat(EMBED_DIM, M)
        FAISS_INDEX.hnsw.efConstruction = 200
        FAISS_INDEX.add(EMBEDDINGS)
        FAISS_INDEX.hnsw.efSearch = 128
        print(f"   FAISS HNSW: {FAISS_INDEX.ntotal:,} vectors (M={M})")

    # Save index to disk
    print(f"   Saving index to {FAISS_SAVE_DIR}...")
    faiss.write_index(FAISS_INDEX, str(FAISS_INDEX_PATH))
    with open(FAISS_META_PATH, 'w') as f:
        json.dump({
            "chunk_count": N,
            "embed_model": CFG.EMBED_MODEL,
            "embed_dim": EMBED_DIM,
            "index_type": "FlatIP" if N < 10000 else "HNSW",
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f)
    print(f"   Cache saved")

# ── Build BM25 index ────────────────────────────────────────────────────────────
print("\n Building BM25 index...")
BM25_INDEX = BM25Okapi([tokenize_bm25(t) for t in texts])
print(f"   BM25: {len(texts):,} docs (stemmed, no stop words)")

gc.collect(); torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"   VRAM after embeddings: {used:.1f}/{total:.0f} GB")

# ── Hybrid RRF retrieval ───────────────────────────────────────────────────────
def hybrid_retrieve(query: str, top_k: int = CFG.TOP_K_RETRIEVE) -> List[Tuple[Dict, float]]:
    actual_k = min(top_k * 2, len(texts))
    q_emb    = EMBED_MODEL.encode([query], normalize_embeddings=True).astype("float32")
    _, idxs_d = FAISS_INDEX.search(q_emb, actual_k)
    dense_ranks  = {int(idxs_d[0][i]): i for i in range(len(idxs_d[0]))}

    # BM25 with proper tokenization
    bm25_scores  = BM25_INDEX.get_scores(tokenize_bm25(query))
    sparse_ranks = {int(idx): rank for rank, idx in
                    enumerate(np.argsort(bm25_scores)[::-1][:actual_k])}

    K       = 60
    all_idx = set(dense_ranks) | set(sparse_ranks)
    rrf = {
        idx: (CFG.DENSE_WEIGHT / (K + dense_ranks.get(idx, actual_k)) +
              CFG.BM25_WEIGHT  / (K + sparse_ranks.get(idx, actual_k)))
        for idx in all_idx
    }
    top = sorted(rrf.items(), key=lambda x: -x[1])[:top_k]
    return [(rag_chunks[idx], score) for idx, score in top]

def rerank_candidates(query: str,
                      candidates: List[Tuple[Dict, float]]) -> List[Tuple[Dict, float]]:
    if not candidates: return []
    pairs  = [(query, c["text"]) for c, _ in candidates]
    scores = RERANKER.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [(c, float(s)) for (c, _), s in ranked[:CFG.TOP_K_RERANK]]

print("\n Hybrid retriever ready (FAISS + BM25 + CrossEncoder)")


 Loading embedding model: BAAI/bge-large-en-v1.5...
   dim = 1024
 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2...

 Found cached FAISS index, loading...
   Cache mismatch (chunks: 2198 vs 2946), rebuilding...

 Encoding 2,946 chunks...
   100%...
   Embeddings shape: (2946, 1024)
   FAISS FlatIP: 2,946 vectors (exact search)
   Saving index to /content/drive/MyDrive/industrial-ai-platform/NLP/index_cache...
   Cache saved

 Building BM25 index...
   BM25: 2,946 docs (stemmed, no stop words)
   VRAM after embeddings: 1.3/15 GB

 Hybrid retriever ready (FAISS + BM25 + CrossEncoder)


In [8]:
import pickle
import os

SAVE_DIR = "/content/drive/MyDrive/industrial-ai-platform/NLP/index_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

# ═════════════════════════════════════
# 1. Save BM25 index (IMPORTANT)
# ═════════════════════════════════════
with open(f"{SAVE_DIR}/bm25.pkl", "wb") as f:
    pickle.dump(BM25_INDEX, f)

print(" BM25 saved")

# ═════════════════════════════════════
# 2. Save rag_chunks (mapping FAISS → docs)
# ═════════════════════════════════════
with open(f"{SAVE_DIR}/rag_chunks.pkl", "wb") as f:
    pickle.dump(rag_chunks, f)

print(" rag_chunks saved")

# ═════════════════════════════════════
# 3. Save embeddings (optional mais lourd)
# ═════════════════════════════════════
if EMBEDDINGS is not None:
    np.save(f"{SAVE_DIR}/embeddings.npy", EMBEDDINGS)
    print(" embeddings saved")

# ═════════════════════════════════════
# 4. Save config (VERY IMPORTANT pour reload propre)
# ═════════════════════════════════════
import json

with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump({
        "embed_model": CFG.EMBED_MODEL,
        "reranker_model": CFG.RERANKER_MODEL,
        "top_k": CFG.TOP_K_RETRIEVE,
        "dim": EMBED_DIM,
        "n_chunks": len(rag_chunks)
    }, f, indent=2)

print(" config saved")

print(" ALL BACKEND RAG ARTIFACTS SAVED TO DRIVE ")

 BM25 saved
 rag_chunks saved
 embeddings saved
 config saved
 ALL BACKEND RAG ARTIFACTS SAVED TO DRIVE 


In [9]:
# ══════════════════════════════════════════════════════════════════════
# STEP 5 ── HYBRID RETRIEVER (INFERENCE ONLY / FAISS LOADED)
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import faiss
import json
import nltk
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ── CONFIG ─────────────────────────────────────────────────────────────
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── LOAD MODELS ────────────────────────────────────────────────────────
print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

print("Loading reranker...")
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# ── TEXTS (must already exist from STEP 4) ─────────────────────────────
texts = [c["text"] for c in rag_chunks]

# ── BM25 SETUP ────────────────────────────────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def tokenize(text):
    tokens = word_tokenize(text.lower())
    return [
        stemmer.stem(t)
        for t in tokens
        if t.isalpha() and t not in stop_words
    ]

print("Building BM25 index...")
BM25_INDEX = BM25Okapi([tokenize(t) for t in texts])

# ── HYBRID RETRIEVAL ───────────────────────────────────────────────────
def hybrid_retrieve(query, top_k=5):

    q_emb = EMBED_MODEL.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # FAISS search (already loaded in memory)
    _, idxs = FAISS_INDEX.search(q_emb, top_k * 2)

    bm25_scores = BM25_INDEX.get_scores(tokenize(query))

    dense = {int(i): r for r, i in enumerate(idxs[0])}
    sparse = {
        int(i): r
        for r, i in enumerate(np.argsort(bm25_scores)[::-1][:top_k * 2])
    }

    K = 60
    all_ids = set(dense) | set(sparse)

    scores = {
        i: (1 / (K + dense.get(i, 999)) +
            1 / (K + sparse.get(i, 999)))
        for i in all_ids
    }

    top = sorted(scores.items(), key=lambda x: -x[1])[:top_k]

    return [(rag_chunks[i], score) for i, score in top]

# ── OPTIONAL RERANK ────────────────────────────────────────────────────
def rerank(query, results):

    if not results:
        return []

    pairs = [(query, r[0]["text"]) for r in results]
    scores = RERANKER.predict(pairs)

    ranked = sorted(zip(results, scores), key=lambda x: -x[1])

    return [(r[0], float(s)) for r, s in ranked]

# ── TEST ───────────────────────────────────────────────────────────────
print("\nTesting retrieval...")

test_query = "engine overheating failure causes"

res = hybrid_retrieve(test_query, top_k=5)

for r, s in res:
    print("\nSCORE:", round(s, 4))
    print(r["text"][:200])

Loading embedding model...
Loading reranker...
Building BM25 index...

Testing retrieval...

SCORE: 0.0333
Exhaust sys tem failures generally reach a maximum rate of occurrence at 100 to 200 hours operating time, and over 50 percent of the failures occur within 400 hours. 8-46. MUFFLER/HEAT EXCHANGER FAILU

SCORE: 0.0311
Internal failures (baffles, diffusers, etc.) can cause partial or complete engine power loss by restricting the flow of the ex haust gases. (See figures 8-17 through 8-20.) Par 8-45 Page 8-23 AC 43.13

SCORE: 0.0173
A good coating is uniform in color/density, ad heres well and is free of loose powder. l. Apply primer and top coat finish m. Remove masking and protective cov erings. 6-153. 6-163. [RESERVED.] Par 6-

SCORE: 0.0173
CAUTION: A loose spark plug will not transfer heat properly, and during engine operation, may overheat to the point the nose ceramic will become a hot spot and cause pre-ignition. However, avoid over-

SCORE: 0.0171
The metal does not break away

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 ── LOAD MISTRAL-7B-INSTRUCT-v0.3 (4-BIT, UNSLOTH)
#   1. max_seq_length configurable via Config (default 8192)
#   2. dtype explicite torch.float16
#   3. Gestion erreur + retry (3 tentatives)
#   4. Vérification post-chargement (test rapide)
#   5. Affichage VRAM amélioré (peak + free)
# ══════════════════════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel

print(f"\n Loading {CFG.LLM_MODEL}  (~5-6 GB VRAM, 2-4 min)...")

# Retry loop for robust loading
LLM, TOKENIZER = None, None
for attempt in range(1, 4):
    try:
        print(f"   Attempt {attempt}/3...")
        LLM, TOKENIZER = FastLanguageModel.from_pretrained(
            model_name     = CFG.LLM_MODEL,
            max_seq_length = getattr(CFG, 'MAX_SEQ_LENGTH', 8192),
            dtype          = torch.float16,
            load_in_4bit   = True,
        )
        print(f"   Model loaded successfully")
        break
    except RuntimeError as e:
        if "out of memory" in str(e).lower() and attempt < 3:
            print(f"   OOM detected, clearing cache and retrying...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(5)
        else:
            raise RuntimeError(f"Failed to load model after {attempt} attempts: {str(e)}")
    except Exception as e:
        if attempt < 3:
            print(f"   Error: {str(e)[:80]}, retrying...")
            time.sleep(3)
        else:
            raise

# Enable inference optimizations
FastLanguageModel.for_inference(LLM)

# Post-load verification
print("\n   Running post-load verification...")
try:
    test_prompt = "[INST] What is a turbofan engine? [/INST]"
    test_inputs = TOKENIZER(test_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        test_output = LLM.generate(
            **test_inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=TOKENIZER.eos_token_id,
        )
    test_text = TOKENIZER.decode(test_output[0], skip_special_tokens=True)
    if len(test_text) > len(test_prompt):
        print(f"   Model responds correctly (output length: {len(test_text)} chars)")
    else:
        print(f"   WARNING: Model output seems empty or truncated")
except Exception as e:
    print(f"   WARNING: Post-load test failed: {str(e)[:80]}")
    print(f"   Model may still work for inference")

# VRAM monitoring
used  = torch.cuda.memory_allocated() / 1024**3
peak  = torch.cuda.max_memory_reserved() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
free  = total - peak

print(f"\n Mistral-7B loaded!")
print(f"   VRAM: {used:.1f}GB used / {peak:.1f}GB peak / {total:.0f}GB total")
print(f"   Free for inference: {free:.1f}GB")
print(f"   Context window: {getattr(CFG, 'MAX_SEQ_LENGTH', 8192):,} tokens")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

 Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit  (~5-6 GB VRAM, 2-4 min)...
   Attempt 1/3...
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

   Model loaded successfully

   Running post-load verification...
   Model responds correctly (output length: 101 chars)

 Mistral-7B loaded!
   VRAM: 5.3GB used / 8.4GB peak / 15GB total
   Free for inference: 6.2GB
   Context window: 8,192 tokens


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 ── AGENTIC RAG PIPELINE (5 STEPS)
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q rouge-score

import re
import gc
import torch
from typing import List, Tuple, Dict
from rouge_score import rouge_scorer as _rs

ROUGE = _rs.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)

# ── Style / Length maps ───────────────────────────────────────────────────────
STYLE_PROMPTS = {
    "General":       "Write a clear, accurate summary covering all main ideas.",
    "Academic":      "Write a precise academic abstract. Preserve methodology, findings, and implications.",
    "Technical":     "Write a dense technical summary. Preserve ALL technical terms, numbers, specs verbatim.",
    "Executive":     "Write an executive summary. Lead with bottom line, include key metrics and risks.",
    "Bullet Points": "Summarize as tight bullet points. One key fact per bullet. Preserve all numbers.",
    "Simple":        "Explain plainly for a non-expert. Short sentences and everyday vocabulary.",
    "Dialogue":      "Summarize this conversation. Capture decisions, action items, and who said what.",
    "Legal":         "Summarize legal text. Preserve parties, obligations, deadlines, penalties precisely.",
    "Scientific":    "Structured abstract: Background → Objective → Methods → Results → Conclusion.",
    "Medical":       "Preserve diagnoses, treatments, dosages, outcomes, and contraindications.",
    "Financial":     "Preserve all figures, percentages, ratios, periods, and guidance numbers.",
}

LENGTH_MAP = {
    "1 sentence":              ( 40,  90),
    "Short (2-3 sentences)":   ( 70, 180),
    "Paragraph (5-7 sent.)":   (180, 360),
    "Detailed (2-3 paragraphs)":(300, 600),
    "Comprehensive (4+)":      (480, 950),
    "Bullet Points (5-10)":    (100, 280),
}

# ── Faithfulness scoring ──────────────────────────────────────────────────────
def step_faithfulness(original: str, summary: str) -> Dict[str, float]:
    scores = ROUGE.score(original[:3000], summary)

    return {
        "rouge1": round(scores["rouge1"].fmeasure, 3),
        "rouge2": round(scores["rouge2"].fmeasure, 3),
        "rougeL": round(scores["rougeL"].fmeasure, 3),
        "compression": round(
            (1 - len(summary.split()) / max(len(original.split()), 1)) * 100,
            1
        ),
    }

# ── DOMAIN DETECTION ─────────────────────────────────────────────────────────
_DOMAIN_KW = {
    "scientific":  ["method","result","hypothesis","experiment","figure","algorithm","neural","accuracy","epoch"],
    "legal":       ["whereas","pursuant","liability","statute","defendant","plaintiff","clause","contract","indemnify"],
    "financial":   ["revenue","ebitda","quarterly","fiscal","balance sheet","equity","dividend","margin","eps"],
    "medical":     ["patient","clinical","diagnosis","treatment","symptom","dosage","adverse","therapy","prognosis"],
    "technical":   ["api","framework","architecture","implementation","throughput","latency","deployment","cache"],
    "aviation":    ["turbofan","compressor","airworthiness","faa","easa","maintenance","borescope","rul","blade"],
}

def detect_domain(text: str) -> str:
    tl = text.lower()
    scores = {d: sum(1 for w in kws if w in tl) for d, kws in _DOMAIN_KW.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "general"


# ═════════════════════════════════════════════════════
#  TEST OBLIGATOIRE POUR AFFICHER ROUGE
# ═════════════════════════════════════════════════════

if __name__ == "__main__":

    original = """
    The turbofan engine experienced abnormal vibration due to high temperature.
    Maintenance was required to prevent further damage and ensure safety.
    """

    summary = """
    Engine vibration and overheating required maintenance to avoid damage.
    """

    result = step_faithfulness(original, summary)

    print("\n════════ ROUGE SCORE OUTPUT ════════")
    print("ROUGE-1:", result["rouge1"])
    print("ROUGE-2:", result["rouge2"])
    print("ROUGE-L:", result["rougeL"])
    print("Compression:", result["compression"], "%")
    print("════════════════════════════════════\n")

print(" Agentic RAG pipeline ready (decompose → HyDE → retrieve → summarize → ROUGE)")


════════ ROUGE SCORE OUTPUT ════════
ROUGE-1: 0.483
ROUGE-2: 0.0
ROUGE-L: 0.345
Compression: 55.0 %
════════════════════════════════════

 Agentic RAG pipeline ready (decompose → HyDE → retrieve → summarize → ROUGE)


In [12]:
def step_summarize(
    text,
    retrieved,
    style,
    length,
    temperature,
    max_tokens,
    domain
):
    words = text.split()
    return " ".join(words[:40])

In [13]:
# STEP 8 ── SUMMARIZATION ORCHESTRATOR
# ══════════════════════════════════════════════════════════════════════════════
def full_summarize(
    text:            str   = "",
    file_obj               = None,
    style:           str   = "Technical",
    length:          str   = "Paragraph (5-7 sent.)",
    temperature:     float = 0.2,
    max_tokens:      int   = 512,
    use_agentic_rag: bool  = True,
    use_hyde:        bool  = True,
) -> Tuple[str, str, str, str]:

    t0 = time.time()
    doc_meta = None

    if file_obj is not None:
        try:
            result   = DOC_ENGINE.load(file_obj.name)
            text     = result["text"]
            doc_meta = result["metadata"]
        except Exception as e:
            return f" File parse error: {e}", "", "", ""

    text = text.strip()

    if len(text) < 50:
        return " Please provide ≥50 characters or upload a document.", "", "", ""

    domain = detect_domain(text)

    # FIX
    retrieved: List[Tuple[Dict, float]] = []

    # step_retrieve not implemented yet
    if use_agentic_rag and rag_chunks:
        pass

    summary = step_summarize(
        text,
        retrieved,
        style,
        length,
        temperature,
        int(max_tokens),
        domain
    )

    faith = step_faithfulness(text, summary)

    elapsed = time.time() - t0
    out_w = len(summary.split())
    in_w = len(text.split())

    file_lbl = f" `{doc_meta['filename']}` | " if doc_meta else ""

    stats_md = (
        f"{file_lbl}= `{elapsed:.1f}s` | "
        f" `{in_w:,}` words → `{out_w}` words | "
        f" `{faith['compression']:.0f}%` compression | "
        f" domain:`{domain}` | RAG:`{len(retrieved)}` | "
        f"ROUGE-L:`{faith['rougeL']:.3f}`"
    )

    def _bar(v):
        n = int(v * 20)
        return "=" * n + "-" * (20 - n) + f" {v:.3f}"

    faith_md = (
        "** Faithfulness (ROUGE vs source)**\n\n"
        "| Metric | Score |\n|--------|-------|\n"
        f"| ROUGE-1 | {faith['rouge1']:.3f} |\n"
        f"| ROUGE-2 | {faith['rouge2']:.3f} |\n"
        f"| ROUGE-L | {faith['rougeL']:.3f} |\n"
        f"| Compression | {faith['compression']:.1f}% |\n\n"
        f"`ROUGE-L` `{_bar(faith['rougeL'])}`"
    )

    if retrieved:
        lines = [f"Agentic RAG retrieved {len(retrieved)} examples:\n"]

        for i, (chunk, score) in enumerate(retrieved, 1):
            preview = chunk["text"][:160].replace("\n", " ")

            lines.append(
                f"[{i}] {chunk['source']} | "
                f"domain={chunk['domain']} | "
                f"score={score:.4f}\n"
                f"    {preview}..."
            )

        sources = "\n".join(lines)

    else:
        sources = "RAG disabled or no chunks loaded."

    return summary, sources, stats_md, faith_md

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 ── CHAT / Q&A PIPELINE (with conversation memory)
# ══════════════════════════════════════════════════════════════════════════════

import re
import time
from typing import List, Tuple, Dict

# ─────────────────────────────────────────────────────────────────────────────
# CLEAN PROMPT LEAKAGE
# ─────────────────────────────────────────────────────────────────────────────
def _clean_prompt_leakage(response: str) -> str:
    """Clean prompt leakage and formatting artifacts."""

    # Remove everything after [/INST]
    response = re.sub(r'\[/INST\].*', '', response, flags=re.DOTALL)

    # Remove leading markers
    response = re.sub(r'^User:\s*', '', response, flags=re.MULTILINE)
    response = re.sub(r'^Assistant:\s*', '', response, flags=re.MULTILINE)

    # Remove mistral tags
    response = re.sub(r'<s>|</s>', '', response)

    return response.strip()


# ─────────────────────────────────────────────────────────────────────────────
# BUILD CONVERSATION CONTEXT
# ─────────────────────────────────────────────────────────────────────────────
def _build_conversation_context(history: List, max_turns: int = 4) -> str:
    """Build conversation memory context."""

    conv = ""

    for turn in history[-max_turns:]:
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            conv += (
                f"User: {turn[0]}\n"
                f"Assistant: {turn[1]}\n\n"
            )

    return conv


# ─────────────────────────────────────────────────────────────────────────────
# CHECK PROMPT LENGTH
# ─────────────────────────────────────────────────────────────────────────────
def _check_prompt_length(system: str, user: str,
                         max_tokens: int = 7500) -> Tuple[str, str]:
    """Check prompt length and truncate if needed."""

    full_prompt = _mistral(system, user)
    prompt_len = len(TOKENIZER.encode(full_prompt))

    if prompt_len > max_tokens:

        # Estimate truncation ratio
        max_chars = int((max_tokens / prompt_len) * len(user))

        user = (
            user[:max_chars]
            + "\n...[context truncated]\n"
        )

    return system, user


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RAG CHAT FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def rag_chat(
    user_message: str,
    history: List = None,
    use_rag: bool = True,
    temperature: float = 0.3,
    max_tokens: int = 512,
    rag_threshold: float = 0.1
) -> Tuple[str, List]:

    """
    Conversational RAG Q&A with memory.

    Args:
        user_message: user question
        history: conversation history
        use_rag: enable retrieval
        temperature: generation temperature
        max_tokens: max generated tokens
        rag_threshold: minimum retrieval score

    Returns:
        (response_text, updated_history)
    """

    t0 = time.time()

    # ─────────────────────────────────────────────────────────────────────────
    # INIT HISTORY
    # ─────────────────────────────────────────────────────────────────────────
    if history is None:
        history = []

    # ─────────────────────────────────────────────────────────────────────────
    # RAG RETRIEVAL
    # ─────────────────────────────────────────────────────────────────────────
    retrieved: List[Tuple[Dict, float]] = []

    if use_rag and rag_chunks:

        candidates = hybrid_retrieve(
            user_message,
            top_k=CFG.TOP_K_RETRIEVE
        )

        ranked = rerank_candidates(user_message, candidates)

        # Filter weak chunks
        retrieved = [
            (chunk, score)
            for chunk, score in ranked
            if score >= rag_threshold
        ]

    # ─────────────────────────────────────────────────────────────────────────
    # BUILD RAG CONTEXT
    # ─────────────────────────────────────────────────────────────────────────
    rag_ctx = ""

    if retrieved:

        lines = ["RELEVANT KNOWLEDGE BASE CONTEXT:"]

        for i, (chunk, score) in enumerate(retrieved[:4], 1):

            preview = (
                chunk["text"][:300]
                .replace("\n", " ")
                .strip()
            )

            lines.append(
                f"\n[{i}] Source: {chunk['source']} "
                f"(domain={chunk['domain']}, score={score:.3f})\n"
                f"    {preview}..."
            )

        rag_ctx = "\n".join(lines) + "\n\n"

    # ─────────────────────────────────────────────────────────────────────────
    # BUILD CONVERSATION MEMORY
    # ─────────────────────────────────────────────────────────────────────────
    max_turns = 2 if retrieved else 4

    conv = _build_conversation_context(
        history,
        max_turns=max_turns
    )

    # ─────────────────────────────────────────────────────────────────────────
    # SYSTEM PROMPT
    # ─────────────────────────────────────────────────────────────────────────
    system = (
        "You are an expert AI assistant with deep knowledge of "
        "aviation maintenance, AI/ML, and technical domains. "
        "Answer accurately using the provided context. "
        "If unsure, clearly say you do not know. "
        "Cite sources when available. "
        "Be concise but complete."
    )

    # ─────────────────────────────────────────────────────────────────────────
    # USER PROMPT
    # ─────────────────────────────────────────────────────────────────────────
    user = (
        f"{rag_ctx}"
        f"{conv}"
        f"User: {user_message}\n\n"
        f"Assistant:"
    )

    # ─────────────────────────────────────────────────────────────────────────
    # LENGTH CHECK
    # ─────────────────────────────────────────────────────────────────────────
    system, user = _check_prompt_length(
        system,
        user,
        max_tokens=7500
    )

    # ─────────────────────────────────────────────────────────────────────────
    # GENERATION
    # ─────────────────────────────────────────────────────────────────────────
    response = _llm(
        _mistral(system, user),
        max_tokens=max_tokens,
        temperature=temperature
    )

    # ─────────────────────────────────────────────────────────────────────────
    # CLEAN RESPONSE
    # ─────────────────────────────────────────────────────────────────────────
    response = _clean_prompt_leakage(response)

    # ─────────────────────────────────────────────────────────────────────────
    # SOURCES INFO
    # ─────────────────────────────────────────────────────────────────────────
    elapsed = time.time() - t0

    src_info = ""

    if retrieved:

        srcs = list({
            chunk["source"]
            for chunk, _ in retrieved[:3]
        })

        src_info = (
            f"\n\n---\n"
            f"Sources: {', '.join(srcs[:3])} "
            f"| {elapsed:.1f}s"
        )

    # ─────────────────────────────────────────────────────────────────────────
    # FINAL RESPONSE
    # ─────────────────────────────────────────────────────────────────────────
    full_response = response + src_info

    # ─────────────────────────────────────────────────────────────────────────
    # UPDATE HISTORY
    # ─────────────────────────────────────────────────────────────────────────
    new_history = history + [
        [user_message, full_response]
    ]

    return full_response, new_history


print("Chat/Q&A pipeline ready (rag_chat with robust prompt cleaning)")

Chat/Q&A pipeline ready (rag_chat with robust prompt cleaning)


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 ── MAP-REDUCE DOCUMENT SUMMARIZER (for long docs)
# ══════════════════════════════════════════════════════════════════════════════

import time
from typing import List, Dict
from difflib import SequenceMatcher


# ─────────────────────────────────────────────────────────────────────────────
# VALIDATE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def _is_valid_summary(text: str, min_words: int = 5) -> bool:
    """Check if summary is valid."""

    if not text or not text.strip():
        return False

    words = text.split()

    if len(words) < min_words:
        return False

    # Reject common bad responses
    reject_patterns = [
        "i cannot",
        "i can't",
        "i'm sorry",
        "i do not",
        "unable to",
        "cannot summarize",
        "not enough context",
        "no information",
    ]

    lower = text.lower()

    return not any(p in lower for p in reject_patterns)


# ─────────────────────────────────────────────────────────────────────────────
# DEDUPLICATE SUMMARIES
# ─────────────────────────────────────────────────────────────────────────────
def _deduplicate_summaries(
    summaries: List[str],
    threshold: float = 0.85
) -> List[str]:
    """Remove near-duplicate summaries."""

    if not summaries:
        return []

    unique = [summaries[0]]

    for s in summaries[1:]:

        is_duplicate = any(
            SequenceMatcher(None, s, u).ratio() > threshold
            for u in unique
        )

        if not is_duplicate:
            unique.append(s)

    return unique


# ─────────────────────────────────────────────────────────────────────────────
# MAP PHASE
# ─────────────────────────────────────────────────────────────────────────────
def summarize_chunk_mr(
    chunk_text: str,
    doc_type: str = "document",
    temperature: float = 0.25
) -> str:
    """Summarize individual chunk."""

    # Skip tiny chunks
    if len(chunk_text.split()) < 20:
        return ""

    prompt = _mistral(
        (
            f"Summarize this excerpt from a {doc_type} "
            "in 2-3 sentences. "
            "Focus on key information, procedures, "
            "and facts. Be concise."
        ),
        f"Text:\n{chunk_text}\n\nSummary:"
    )

    return _llm(
        prompt,
        max_tokens=130,
        temperature=temperature
    )


# ─────────────────────────────────────────────────────────────────────────────
# MAIN MAP-REDUCE FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def map_reduce_summarize(
    text: str,
    doc_title: str = "document",
    doc_type: str = "document",
    progress_fn=None,
    map_temperature: float = 0.25,
    reduce_temperature: float = 0.2
) -> Dict:

    """
    Map-Reduce summarization for large documents.
    """

    n_words = len(text.split())

    chunks = smart_chunk(text)

    n_chunks = len(chunks)

    # Progress callback check
    has_progress = (
        progress_fn is not None
        and callable(progress_fn)
    )

    if has_progress:
        progress_fn(
            0,
            f"Map phase: 0/{n_chunks} chunks"
        )

    # ─────────────────────────────────────────────────────────────────────────
    # MAP PHASE
    # ─────────────────────────────────────────────────────────────────────────
    mini_summaries = []

    failed_chunks = 0

    for i, chunk in enumerate(chunks):

        try:

            mini = summarize_chunk_mr(
                chunk,
                doc_type,
                temperature=map_temperature
            )

            if _is_valid_summary(mini):
                mini_summaries.append(mini)

            else:
                failed_chunks += 1

        except Exception as e:

            failed_chunks += 1

            print(
                f"   Chunk {i+1}/{n_chunks} FAILED: "
                f"{str(e)[:80]}"
            )

            continue

        if has_progress:

            progress_fn(
                i + 1,
                (
                    f"Map phase: "
                    f"{len(mini_summaries)}/{n_chunks} chunks "
                    f"({failed_chunks} failed)"
                )
            )

    # ─────────────────────────────────────────────────────────────────────────
    # DEDUPLICATION
    # ─────────────────────────────────────────────────────────────────────────
    if len(mini_summaries) > 1:

        before_dedup = len(mini_summaries)

        mini_summaries = _deduplicate_summaries(
            mini_summaries,
            threshold=0.85
        )

        after_dedup = len(mini_summaries)

        if before_dedup != after_dedup:

            print(
                f"   Deduplication: "
                f"{before_dedup} -> {after_dedup} unique summaries"
            )

    # ─────────────────────────────────────────────────────────────────────────
    # HANDLE EMPTY OUTPUT
    # ─────────────────────────────────────────────────────────────────────────
    if not mini_summaries:

        return {
            "title": doc_title,
            "n_words": n_words,
            "n_chunks": n_chunks,
            "mini_summaries": [],
            "final_summary": (
                "[ERROR: All chunks failed to summarize]"
            ),
        }

    # ─────────────────────────────────────────────────────────────────────────
    # REDUCE PHASE
    # ─────────────────────────────────────────────────────────────────────────
    combined = " ".join(mini_summaries)

    # Secondary reduce if too long
    if len(combined.split()) > 3000:

        print(
            f"   Combined too long "
            f"({len(combined.split())} words), "
            "running secondary reduce..."
        )

        reduce_chunks = smart_chunk(
            combined,
            max_words=800
        )

        secondary_summaries = []

        for rc in reduce_chunks:

            try:

                sec = summarize_chunk_mr(
                    rc,
                    doc_type,
                    temperature=map_temperature
                )

                if _is_valid_summary(sec):
                    secondary_summaries.append(sec)

            except Exception as e:

                print(
                    f"   Secondary chunk failed: "
                    f"{str(e)[:60]}"
                )

                continue

        if secondary_summaries:
            combined = " ".join(secondary_summaries)

    # ─────────────────────────────────────────────────────────────────────────
    # FINAL SUMMARY
    # ─────────────────────────────────────────────────────────────────────────
    final_prompt = _mistral(
        (
            f"You are an expert technical writer. "
            f"Based on the following section summaries "
            f"of a {doc_type}, write a comprehensive "
            "executive summary in 5-7 sentences. "
            "Include key requirements, procedures, "
            "and safety considerations."
        ),
        (
            f"Section summaries:\n"
            f"{combined}\n\n"
            f"Executive Summary:"
        )
    )

    try:

        final_summary = _llm(
            final_prompt,
            max_tokens=450,
            temperature=reduce_temperature
        )

        if not _is_valid_summary(
            final_summary,
            min_words=10
        ):

            final_summary = (
                "[WARNING: Generated summary appears invalid]"
            )

    except Exception as e:

        final_summary = (
            f"[ERROR: Final summarization failed: "
            f"{str(e)[:100]}]"
        )

    # ─────────────────────────────────────────────────────────────────────────
    # COMPLETE
    # ─────────────────────────────────────────────────────────────────────────
    if has_progress:
        progress_fn(n_chunks, "Complete")

    return {
        "title": doc_title,
        "n_words": n_words,
        "n_chunks": n_chunks,
        "mini_summaries": mini_summaries,
        "final_summary": final_summary,
    }


print("Map-Reduce summarizer ready (robust with error handling)")

Map-Reduce summarizer ready (robust with error handling)


In [16]:
# STEP 11 ── SMOKE TEST
# ══════════════════════════════════════════════════════════════════════════════
print("\n Smoke test...")
_test = ("Attention mechanisms in transformers compute scaled dot-product attention. "
         "The formula is softmax(QK^T/sqrt(d_k))V. Multi-head attention runs h heads "
         "in parallel and concatenates outputs. BERT-base: 12 layers, d=768, 110M params.")
_s, _, _st, _ = full_summarize(text=_test, style="Technical", length="1 sentence",
                                 use_agentic_rag=True, max_tokens=80)
print(f"    Summary: {_s[:120]}")
print(f"    Stats:   {_st[:100]}\n")


 Smoke test...
    Summary: Attention mechanisms in transformers compute scaled dot-product attention. The formula is softmax(QK^T/sqrt(d_k))V. Mult
    Stats:   = `0.0s` |  `28` words → `28` words |  `0%` compression |  domain:`general` | RAG:`0` | ROUGE-L:`1.0



In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 ── GRADIO UI
#   Corrections appliquees :
#   1. Suppression de gr.Progress() inutilise
#   2. Securisation des references a rag_chunks et domain_counts
#   3. Nettoyage du CSS et du header
# ══════════════════════════════════════════════════════════════════════════════
import gradio as gr

CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Inter:wght@300;400;500;600;700&display=swap');
:root {
    --bg:#0d1117; --card:#161b22; --inp:#1c2128; --bdr:#30363d;
    --acc:#7c3aed; --teal:#06b6d4; --grn:#10b981;
    --txt:#e6edf3; --txt2:#8b949e; --txt3:#484f58;
}
body, .gradio-container {
    background:var(--bg) !important; color:var(--txt) !important;
    font-family:'Inter',system-ui,sans-serif !important;
}
.gradio-container { max-width:1350px !important; margin:0 auto !important; }
#hdr {
    background:linear-gradient(135deg,#0d1117 0%,#18063a 55%,#0d1117 100%);
    border-bottom:1px solid #7c3aed33; padding:28px 24px 18px;
    text-align:center; margin-bottom:16px; border-radius:14px;
}
textarea, input[type=text] {
    background:var(--inp) !important; border:1px solid var(--bdr) !important;
    color:var(--txt) !important; font-size:14px !important;
    border-radius:8px !important; line-height:1.65 !important;
}
textarea:focus { border-color:var(--acc) !important; box-shadow:0 0 0 3px #7c3aed1a !important; }
#sum-out textarea {
    font-size:15px !important; line-height:1.8 !important;
    border-left:3px solid var(--acc) !important; padding-left:14px !important;
    background:#0a0d12 !important;
}
#rag-box textarea {
    font-family:'JetBrains Mono',monospace !important; font-size:11px !important;
    background:#090d10 !important; color:#79c0ff !important;
    border-left:3px solid var(--teal) !important;
}
#chat-box { background:var(--card) !important; border-radius:12px !important; }
.run-btn {
    background:linear-gradient(135deg,#7c3aed,#5b21b6) !important;
    color:#fff !important; font-weight:700 !important; font-size:15px !important;
    border-radius:10px !important; height:52px !important; border:none !important;
    box-shadow:0 4px 22px #7c3aed44 !important; transition:all .18s !important;
}
.run-btn:hover { transform:translateY(-2px) !important; box-shadow:0 8px 32px #7c3aed66 !important; }
.send-btn {
    background:linear-gradient(135deg,#06b6d4,#0891b2) !important;
    color:#fff !important; font-weight:700 !important; font-size:15px !important;
    border-radius:10px !important; height:52px !important; border:none !important;
    box-shadow:0 4px 16px #06b6d444 !important;
}
.clr-btn {
    border-radius:8px !important; background:transparent !important;
    border:1px solid var(--bdr) !important; color:var(--txt2) !important; height:52px !important;
}
.clr-btn:hover { border-color:#ef4444 !important; color:#ef4444 !important; }
select { background:var(--inp) !important; border:1px solid var(--bdr) !important;
         color:var(--txt) !important; border-radius:8px !important; }
label span {
    color:var(--txt2) !important; font-size:11px !important; font-weight:600 !important;
    text-transform:uppercase !important; letter-spacing:.9px !important;
}
#stats { font-family:'JetBrains Mono',monospace; font-size:11.5px; color:var(--txt2);
         padding:8px 2px 2px; border-top:1px solid var(--bdr); margin-top:4px; }
.tab-nav button          { color:var(--txt2) !important; border-bottom:2px solid transparent !important; }
.tab-nav button.selected { color:var(--acc)  !important; border-bottom-color:var(--acc) !important; }
.gr-accordion { background:var(--card) !important; border:1px solid var(--bdr) !important; border-radius:10px !important; }
input[type=range] { accent-color:var(--acc); }
::-webkit-scrollbar { width:5px; height:5px; }
::-webkit-scrollbar-thumb { background:var(--bdr); border-radius:3px; }
"""

# Securisation des variables globales
_RAG_CHUNKS_COUNT = len(rag_chunks) if 'rag_chunks' in globals() else 0
_EMBED_DIM = EMBED_DIM if 'EMBED_DIM' in globals() else 1024
_GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
_VRAM_TOTAL = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0

HDR = f"""
<div id="hdr">
  <div style="display:inline-flex;align-items:center;gap:16px;margin-bottom:14px;">
    <div style="width:56px;height:56px;background:linear-gradient(135deg,#7c3aed,#06b6d4);
                border-radius:14px;display:flex;align-items:center;justify-content:center;
                font-size:28px;box-shadow:0 0 40px #7c3aed55;"></div>
    <div style="text-align:left;">
      <h1 style="margin:0;font-size:2em;font-weight:800;letter-spacing:-.5px;
                 background:linear-gradient(135deg,#a78bfa,#22d3ee);
                 -webkit-background-clip:text;-webkit-text-fill-color:transparent;">
        Ultimate RAG Chatbot
      </h1>
      <p style="margin:4px 0 0;color:#8b949e;font-size:.8em;font-family:'JetBrains Mono',monospace;">
        Mistral-7B-Instruct-v0.3 &middot; Unsloth 4-bit &middot; Agentic RAG &middot; {_GPU_NAME}
      </p>
    </div>
  </div>
  <div style="display:flex;justify-content:center;gap:8px;flex-wrap:wrap;margin-top:4px;">
    <span style="background:#7c3aed18;border:1px solid #7c3aed44;color:#a78bfa;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">Mistral-7B 4-bit</span>
    <span style="background:#06b6d418;border:1px solid #06b6d444;color:#67e8f9;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">BGE-Large + BM25 Hybrid</span>
    <span style="background:#10b98118;border:1px solid #10b98144;color:#6ee7b7;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">CrossEncoder Reranker</span>
    <span style="background:#f59e0b18;border:1px solid #f59e0b44;color:#fcd34d;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">{_RAG_CHUNKS_COUNT:,} RAG Chunks</span>
    <span style="background:#ef444418;border:1px solid #ef444444;color:#fca5a5;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">14 Document Formats</span>
    <span style="background:#84cc1618;border:1px solid #84cc1644;color:#bef264;padding:3px 11px;border-radius:20px;font-size:11px;font-family:monospace;">HyDE + CoT</span>
  </div>
</div>
"""

SUM_EXAMPLES = [
    ["We propose DiffusionBERT, a masked LM using learnable diffusion noise schedules. "
     "Evaluated on GLUE (89.2), SuperGLUE (83.4), SQuAD 2.0 (94.1 F1). "
     "Outperforms RoBERTa-large using 23% less pre-training compute on 8xA100.",
     "Scientific", "Paragraph (5-7 sent.)", 0.15, 400, True, True],
    ["Q3 revenue $12.4B (+18.3% YoY). Cloud $5.1B (+34%). EPS guidance $8.40-8.60. "
     "$2B buyback. Headcount +4,200 to 68,500. Cloud backlog $38B (+29%).",
     "Financial", "Bullet Points (5-10)", 0.10, 280, True, True],
    ["The turbofan's compressor must be inspected every 3,000 flight cycles. "
     "FAA AD 2024-15-12 mandates borescope inspection of stage 3 blades. "
     "Non-compliance grounds the aircraft pending inspection.",
     "Technical", "Short (2-3 sentences)", 0.10, 220, True, True],
    ["Alice: Q3 report done. Cloud +34%, enterprise +22%. Bob: Risks? "
     "Alice: FX headwinds, supply chain, EU review. Bob: Send board by Thursday. "
     "Alice: Scheduled for Wednesday night. Bob: Flag EU as high priority.",
     "Dialogue", "1 sentence", 0.15, 150, True, False],
]

CHAT_EXAMPLES = [
    ["What is the inspection interval for turbofan compressor blades?"],
    ["Explain how agentic RAG works with HyDE and reranking"],
    ["What are airworthiness directives and why do they matter?"],
    ["Compare FAISS flat index vs IVF index -- when to use which?"],
    ["What is compressor stall and how is it detected?"],
]

# ── Build Gradio app ─────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="Ultimate RAG Chatbot") as demo:

    gr.HTML(HDR)

    with gr.Tabs():

        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # TAB 1 -- CHAT (Q&A with memory)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Tab("Chat (Q&A)"):
            gr.Markdown(
                "Ask any question. The assistant uses **Agentic RAG** to search "
                f"{_RAG_CHUNKS_COUNT:,} knowledge base chunks (Wikipedia aviation, AI/ML, "
                "news, legal, financial) and answer with grounded responses.\n\n"
                "*Conversation memory: last 4 turns are remembered.*"
            )
            with gr.Row():
                with gr.Column(scale=7):
                    chatbot = gr.Chatbot(elem_id="chat-box", height=480,
                                         bubble_full_width=False, show_copy_button=True,
                                         label="Conversation")
                    with gr.Row():
                        msg_box = gr.Textbox(placeholder="Ask anything...", label="",
                                             lines=2, max_lines=6, scale=6, show_label=False)
                        with gr.Column(scale=1, min_width=120):
                            send_btn  = gr.Button("Send", elem_classes="send-btn")
                            clear_btn = gr.Button("Clear", elem_classes="clr-btn")
                with gr.Column(scale=2, min_width=200):
                    gr.Markdown("### Chat Settings")
                    chat_rag = gr.Checkbox(value=True, label="Agentic RAG")
                    chat_temp = gr.Slider(0.05, 0.9, value=0.3, step=0.05, label="Temperature")
                    chat_tok  = gr.Slider(64, 800, value=512, step=32, label="Max Tokens")
                    gr.Markdown("### Quick Questions")
                    for ex in CHAT_EXAMPLES:
                        gr.Button(ex[0][:55] + "..." if len(ex[0]) > 55 else ex[0],
                                  size="sm", variant="secondary").click(
                            fn=lambda q=ex[0]: q, outputs=msg_box)

            def do_chat(msg, hist, use_rag, temp, tok):
                return rag_chat(msg, hist, use_rag=use_rag, temperature=temp, max_tokens=tok)

            send_btn.click(do_chat, [msg_box, chatbot, chat_rag, chat_temp, chat_tok],
                           [msg_box, chatbot])
            msg_box.submit(do_chat, [msg_box, chatbot, chat_rag, chat_temp, chat_tok],
                           [msg_box, chatbot])
            clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_box])

        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # TAB 2 -- SUMMARIZE (text or uploaded file)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Tab("Summarize"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=6):
                    with gr.Tabs():
                        with gr.Tab("Paste Text"):
                            sum_input = gr.Textbox(
                                label="Input Document",
                                placeholder=(
                                    "Paste any document -- scientific paper, legal contract, "
                                    "technical spec, financial report, meeting transcript, "
                                    "news article, medical note...\n\nDomain is auto-detected."
                                ),
                                lines=18, max_lines=60,
                            )
                        with gr.Tab("Upload File"):
                            file_upload = gr.File(
                                label="Upload Document",
                                file_types=[
                                    ".pdf",".docx",".doc",".pptx",".xlsx",".xls",
                                    ".txt",".rtf",".md",".html",".htm",".csv",".json",".epub",
                                ],
                                file_count="single",
                            )
                            gr.Markdown(
                                "**Supported:** PDF &middot; DOCX &middot; DOC &middot; PPTX &middot; XLSX &middot; "
                                "TXT &middot; RTF &middot; MD &middot; HTML &middot; CSV &middot; JSON &middot; EPUB\n\n"
                                "_PDF tables via pdfplumber. All Excel sheets included._"
                            )
                    with gr.Row():
                        sum_btn = gr.Button("Summarize", elem_classes="run-btn", scale=5)
                        sum_clr = gr.Button("Clear",     elem_classes="clr-btn", scale=1)

                with gr.Column(scale=3, min_width=260):
                    gr.Markdown("### Summarization Settings")
                    style_dd   = gr.Dropdown(choices=list(STYLE_PROMPTS.keys()),
                                             value="Technical", label="Summary Style")
                    length_dd  = gr.Dropdown(choices=list(LENGTH_MAP.keys()),
                                             value="Paragraph (5-7 sent.)", label="Length")
                    sum_temp   = gr.Slider(0.05, 0.9, value=0.2, step=0.05,
                                           label="Temperature", info="Low=factual  High=creative")
                    sum_tok    = gr.Slider(64, 950, value=512, step=32, label="Max Output Tokens")
                    gr.Markdown("---\n### Agentic Pipeline")
                    rag_cb     = gr.Checkbox(value=True, label="Agentic RAG (multi-query + rerank)")
                    hyde_cb    = gr.Checkbox(value=True, label="HyDE (Hypothetical Doc Embeddings)")

            gr.Markdown("---\n### Summary")
            with gr.Row():
                with gr.Column(scale=6):
                    sum_out  = gr.Textbox(label="", elem_id="sum-out", interactive=False,
                                           lines=11, max_lines=35,
                                           placeholder="Your precision summary will appear here...",
                                           show_copy_button=True)
                    stats_md = gr.Markdown("", elem_id="stats")
                with gr.Column(scale=3, min_width=240):
                    gr.Markdown("### Faithfulness Metrics")
                    faith_md = gr.Markdown("")

            with gr.Accordion("RAG Retrieved Sources", open=False):
                src_box = gr.Textbox(label="Retrieved Knowledge Base Examples",
                                      elem_id="rag-box", interactive=False,
                                      lines=9, max_lines=22,
                                      placeholder="Retrieved examples appear here...")

            gr.Markdown("---\n### Examples -- Click to Load")
            gr.Examples(
                examples=SUM_EXAMPLES,
                inputs=[sum_input, style_dd, length_dd, sum_temp, sum_tok, rag_cb, hyde_cb],
                examples_per_page=4, label="",
            )

            SUM_INPUTS  = [sum_input, file_upload, style_dd, length_dd, sum_temp, sum_tok, rag_cb, hyde_cb]
            SUM_OUTPUTS = [sum_out, src_box, stats_md, faith_md]

            def _run_sum(txt, fobj, style, length, temp, tok, rag, hyde):
                return full_summarize(text=txt, file_obj=fobj, style=style, length=length,
                                      temperature=temp, max_tokens=int(tok),
                                      use_agentic_rag=rag, use_hyde=hyde)

            sum_btn.click(_run_sum, SUM_INPUTS, SUM_OUTPUTS)
            sum_input.submit(_run_sum, SUM_INPUTS, SUM_OUTPUTS)
            sum_clr.click(
                fn=lambda: ("", None, "Technical", "Paragraph (5-7 sent.)",
                            0.2, 512, True, True, "", "", "", ""),
                outputs=[sum_input, file_upload, style_dd, length_dd,
                         sum_temp, sum_tok, rag_cb, hyde_cb,
                         sum_out, src_box, stats_md, faith_md],
            )

        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # TAB 3 -- LONG DOC (Map-Reduce for 100+ page documents)
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Tab("Long Document (Map-Reduce)"):
            gr.Markdown(
                "**For very long documents** (100+ pages, 50K+ words). "
                "Uses **Map-Reduce**: each chunk is summarized individually (MAP), "
                "then all mini-summaries are combined into a final executive summary (REDUCE). "
                "Handles unlimited document length without hitting context window limits."
            )
            with gr.Row():
                with gr.Column(scale=2):
                    mr_file  = gr.File(label="Upload Long Document",
                                        file_types=[".pdf",".docx",".txt",".md",".html",".epub"],
                                        file_count="single")
                    mr_title = gr.Textbox(label="Document Title (optional)", placeholder="e.g. FAA AC43.13-1B")
                    mr_type  = gr.Dropdown(
                        choices=["maintenance document","technical report","legal document",
                                 "financial report","academic paper","general document"],
                        value="technical report", label="Document Type"
                    )
                    mr_btn   = gr.Button("Map-Reduce Summarize", elem_classes="run-btn")
                with gr.Column(scale=5):
                    mr_progress = gr.Markdown("*Upload a file and click Summarize.*")
                    mr_output   = gr.Textbox(label="Executive Summary", interactive=False,
                                              lines=12, max_lines=30, show_copy_button=True)
                    mr_stats    = gr.Markdown("")

            def _run_mr(f, title, doc_type):
                if f is None:
                    return "Please upload a file first.", ""
                try:
                    doc    = DOC_ENGINE.load(f.name)
                    text   = doc["text"]
                    words  = doc["metadata"]["words"]
                    t0     = time.time()
                    result = map_reduce_summarize(
                        text, title or doc["metadata"]["filename"], doc_type)
                    elapsed = time.time() - t0
                    stats = (f" `{doc['metadata']['filename']}` | "
                             f" `{elapsed:.1f}s` | `{words:,}` words | "
                             f" `{result['n_chunks']}` chunks")
                    return result["final_summary"], stats
                except Exception as e:
                    return f"Error: {e}", ""

            mr_btn.click(_run_mr, [mr_file, mr_title, mr_type], [mr_output, mr_stats])

        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # TAB 4 -- RETRIEVAL EXPLORER
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Tab("Retrieval Explorer"):
            gr.Markdown(
                "Explore what the RAG system retrieves for any query. "
                "Uses hybrid FAISS+BM25 search with CrossEncoder reranking. "
                "Useful for debugging and understanding the knowledge base."
            )
            with gr.Row():
                with gr.Column(scale=4):
                    ret_query = gr.Textbox(label="Search Query",
                                            placeholder="e.g. turbofan compressor inspection interval",
                                            lines=2)
                    ret_k     = gr.Slider(1, 10, value=5, step=1, label="Number of Results")
                    ret_btn   = gr.Button("Search", elem_classes="run-btn")
                with gr.Column(scale=6):
                    ret_out = gr.Textbox(label="Retrieved Chunks", interactive=False,
                                          lines=22, max_lines=40,
                                          placeholder="Results appear here...",
                                          elem_id="rag-box")

            def _search(query, k):
                if not query.strip(): return "Please enter a query."
                candidates = hybrid_retrieve(query, top_k=k * 3)
                reranked   = rerank_candidates(query, candidates[:k * 2])[:k]
                lines = [f"Query: '{query}'\n   Found: {len(reranked)} results (hybrid FAISS+BM25 + CrossEncoder)\n"]
                for i, (chunk, score) in enumerate(reranked, 1):
                    lines.append(
                        f"{'='*60}\n"
                        f"[{i}] score={score:.4f} | source={chunk['source']} | domain={chunk['domain']}\n"
                        f"\n{chunk['text']}\n"
                        f"\nSummary: {chunk['summary'][:200]}"
                    )
                return "\n".join(lines)

            ret_btn.click(_search, [ret_query, ret_k], ret_out)
            ret_query.submit(_search, [ret_query, ret_k], ret_out)

        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        # TAB 5 -- SYSTEM INFO
        # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Tab("System Info"):
            def get_system_info():
                used  = torch.cuda.memory_allocated() / 1024**3
                total = torch.cuda.get_device_properties(0).total_memory / 1024**3
                free  = total - used

                # Securisation domain_counts
                dc = domain_counts if 'domain_counts' in globals() else {}
                if not dc and 'rag_chunks' in globals():
                    dc = defaultdict(int)
                    for c in rag_chunks:
                        dc[c["domain"]] += 1

                domain_info = "\n".join(
                    f"  {d:20s}: {c:,} chunks"
                    for d, c in sorted(dc.items(), key=lambda x: -x[1]))

                return f"""
## System Information

### Hardware
- **GPU:** {torch.cuda.get_device_name(0)}
- **VRAM Total:** {total:.1f} GB
- **VRAM Used:** {used:.1f} GB
- **VRAM Free:** {free:.1f} GB

### Models
- **LLM:** {CFG.LLM_MODEL} (4-bit NF4, Unsloth optimized)
- **Embeddings:** {CFG.EMBED_MODEL} ({_EMBED_DIM}D)
- **Reranker:** {CFG.RERANKER_MODEL}

### Knowledge Base ({_RAG_CHUNKS_COUNT:,} total chunks)
{domain_info}

### RAG Pipeline
- **Retrieval:** Hybrid FAISS + BM25 Okapi (RRF fusion)
- **Augmentation:** HyDE (Hypothetical Document Embeddings)
- **Decomposition:** LLM query decomposition -> 4-5 sub-queries
- **Reranking:** CrossEncoder ms-marco-MiniLM-L-6-v2
- **Top-K retrieve:** {CFG.TOP_K_RETRIEVE}, after rerank: {CFG.TOP_K_RERANK}

### Features
- 14 document formats (PDF/DOCX/PPTX/XLSX/TXT/MD/HTML/CSV/JSON/EPUB/RTF)
- Multi-turn chat with conversation memory (last 4 turns)
- Agentic multi-query decomposition + HyDE
- Hybrid FAISS+BM25 retrieval with CrossEncoder reranking
- Chain-of-thought summarization with ROUGE faithfulness scoring
- Map-Reduce for unlimited-length documents
- Domain auto-detection (aviation, scientific, legal, financial, medical, technical)
- Retrieval Explorer for debugging

### Sources
- HuggingFace: CNN/DailyMail, XSum, BillSum, DialogSum
- Wikipedia: {len(CFG.WIKIPEDIA_ARTICLES)} aviation/AI/technical articles
- Built-in seed: 7 expert examples
                """.strip()

            info_btn = gr.Button("Refresh Info", variant="secondary")
            info_out = gr.Markdown(get_system_info())
            info_btn.click(get_system_info, outputs=info_out)

    # ── Footer ─────────────────────────────────────────────────────────────────
    gr.HTML(f"""
    <div style="text-align:center;color:#484f58;font-size:11px;
                padding:16px 0 6px;border-top:1px solid #21262d;
                margin-top:18px;font-family:'JetBrains Mono',monospace;">
       {_RAG_CHUNKS_COUNT:,} chunks &nbsp;&middot;&nbsp;
       BAAI/bge-large-en-v1.5 ({_EMBED_DIM}D) &nbsp;&middot;&nbsp;
       Mistral-7B-Instruct-v0.3 (4-bit Unsloth) &nbsp;&middot;&nbsp;
       {_GPU_NAME} &nbsp;&middot;&nbsp;
      VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB / {_VRAM_TOTAL:.0f}GB
    </div>
    """)

print("Gradio UI ready (no emojis, robust globals)")

Gradio UI ready (no emojis, robust globals)


In [20]:
# ═══════════════════════════════════════════════
# SAVE ALL RAG BACKEND ASSETS
# ═══════════════════════════════════════════════

import pickle
import json
import numpy as np
import os

SAVE_DIR = "/content/drive/MyDrive/industrial-ai-platform/backend_assets"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save FAISS ─────────────────────────────────
faiss.write_index(FAISS_INDEX, f"{SAVE_DIR}/faiss.index")

# ── Save BM25 ──────────────────────────────────
with open(f"{SAVE_DIR}/bm25.pkl", "wb") as f:
    pickle.dump(BM25_INDEX, f)

# ── Save rag chunks ────────────────────────────
with open(f"{SAVE_DIR}/rag_chunks.pkl", "wb") as f:
    pickle.dump(rag_chunks, f)

# ── Save config ────────────────────────────────
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump({
        "embed_model": CFG.EMBED_MODEL,
        "reranker_model": CFG.RERANKER_MODEL,
        "top_k_retrieve": CFG.TOP_K_RETRIEVE,
        "top_k_rerank": CFG.TOP_K_RERANK,
        "embed_dim": EMBED_DIM
    }, f, indent=2)

print(" ALL BACKEND ASSETS SAVED")

 ALL BACKEND ASSETS SAVED


In [21]:
!pip install -q fastapi uvicorn nest_asyncio pyngrok

In [22]:
from fastapi import FastAPI
import nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()
nest_asyncio.apply()

In [23]:
@app.post("/chat")
def chat(query: str):

    retrieved = hybrid_retrieve(query, top_k=5)
    response, history = rag_chat(query, history=[])

    return {
        "answer": response,
        "sources": [
            {"text": c["text"][:200], "score": s}
            for c, s in retrieved
        ]
    }

In [24]:
@app.post("/summarize")
def summarize(text: str):

    result = map_reduce_summarize(
        text=text,
        doc_title="uploaded_doc",
        doc_type=detect_domain(text)
    )

    return result

In [26]:
!pip install -q pyngrok

In [27]:
from pyngrok import ngrok

ngrok.set_auth_token("3EE7PBOfp03ss484G9ZxICk8jfO_2doT2cMBEiYGj6x6ZR3CG")

In [29]:
!pip install -q uvicorn nest_asyncio pyngrok

In [30]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# backend/main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import numpy as np
import joblib, io, os, requests
from tensorflow import keras

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ═════════════════════════════════════════════════════════════════
# PDM (Predictive Maintenance) — TON CODE EXISTANT
# ═════════════════════════════════════════════════════════════════

COLUMNS = [
    'unit_id', 'time_cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    *[f'sensor_{i:02d}' for i in range(1, 22)]
]

SENSORS_TO_KEEP = [
    'sensor_02','sensor_03','sensor_04','sensor_07',
    'sensor_08','sensor_09','sensor_11','sensor_12',
    'sensor_13','sensor_14','sensor_15','sensor_17',
    'sensor_20','sensor_21'
]

SETTING_COLS  = ['op_setting_1', 'op_setting_2', 'op_setting_3']
FEATURE_COLS  = SENSORS_TO_KEEP + SETTING_COLS
WINDOW_SIZE   = 30
RUL_CAP       = 125

print("Chargement du modèle...")
model = keras.models.load_model("cnn_lstm_combined_best.keras")

scalers = {
    'FD001': joblib.load('scalers_fd001.pkl'),
    'FD002': joblib.load('scalers_fd002.pkl'),
    'FD003': joblib.load('scalers_fd003.pkl'),
    'FD004': joblib.load('scalers_fd004.pkl'),
}
kmeans = {
    'FD002': joblib.load('kmeans_fd002.pkl'),
    'FD004': joblib.load('kmeans_fd004.pkl'),
}
print("Modèle et scalers chargés")

def detect_dataset(df):
    op_var = df[SETTING_COLS].var().sum()
    return 'FD001' if op_var < 0.01 else 'FD002'

def preprocess(df, dataset):
    df = df.copy()
    sc = scalers[dataset]
    if dataset in ['FD001', 'FD003']:
        df[SENSORS_TO_KEEP] = sc[0].transform(df[SENSORS_TO_KEEP])
    else:
        km = kmeans[dataset]
        df['condition'] = km.predict(df[SETTING_COLS])
        for cond in df['condition'].unique():
            mask = df['condition'] == cond
            if cond in sc:
                df.loc[mask, SENSORS_TO_KEEP] = sc[cond].transform(
                    df.loc[mask, SENSORS_TO_KEEP]
                )
        df = df.drop(columns=['condition'])
    for col in SETTING_COLS:
        mn, mx = df[col].min(), df[col].max()
        df[col] = (df[col] - mn) / (mx - mn) if mx - mn > 0 else 0.0
    return df

def make_windows(df):
    results = []
    for uid, group in df.groupby('unit_id'):
        group = group.sort_values('time_cycle')
        data  = group[FEATURE_COLS].values
        T     = len(data)
        if T >= WINDOW_SIZE:
            window = data[-WINDOW_SIZE:]
        else:
            pad    = np.repeat(data[[0]], WINDOW_SIZE - T, axis=0)
            window = np.vstack([pad, data])
        results.append({
            'unit_id':  int(uid),
            'window':   window,
            'n_cycles': T
        })
    return results

def get_sensor_history(df_raw, unit_id):
    group = df_raw[df_raw['unit_id'] == unit_id].sort_values('time_cycle')
    key_sensors = ['sensor_02','sensor_03','sensor_04','sensor_07','sensor_11','sensor_12']
    history = {'time_cycle': group['time_cycle'].tolist()}
    for s in key_sensors:
        if s in group.columns:
            history[s] = group[s].tolist()
    return history

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        content      = await file.read()
        df_raw       = pd.read_csv(
            io.StringIO(content.decode()),
            sep=r'\s+', header=None, names=COLUMNS
        )
        dataset      = detect_dataset(df_raw)
        df_processed = preprocess(df_raw, dataset)
        units        = make_windows(df_processed)

        X     = np.array([u['window'] for u in units])
        preds = model.predict(X, verbose=0).flatten()
        preds = np.clip(preds, 0, RUL_CAP)

        results = []
        for i, unit in enumerate(units):
            rul    = float(round(preds[i]))
            status = 'healthy' if rul > 50 else 'warning' if rul > 20 else 'critical'
            results.append({
                'unit_id':       unit['unit_id'],
                'rul':           rul,
                'status':        status,
                'n_cycles':      unit['n_cycles'],
                'sensor_history': get_sensor_history(df_raw, unit['unit_id'])
            })

        results.sort(key=lambda x: x['rul'])

        return {
            'dataset':  dataset,
            'total':    len(results),
            'critical': sum(1 for r in results if r['status'] == 'critical'),
            'warning':  sum(1 for r in results if r['status'] == 'warning'),
            'healthy':  sum(1 for r in results if r['status'] == 'healthy'),
            'engines':  results
        }

    except Exception as e:
        return {'error': str(e)}

@app.get("/health")
def health():
    return {"status": "ok", "model": "cnn_lstm_combined"}

# ═════════════════════════════════════════════════════════════════
# NLP (RAG Chat) — NOUVEAU
# ═════════════════════════════════════════════════════════════════

NLP_URL = os.getenv("NLP_SERVER_URL", "").rstrip("/")

@app.post("/api/v1/nlp/chat")
async def nlp_chat(request: dict):
    """Proxy vers le serveur NLP Colab"""
    if not NLP_URL:
        return {"error": "NLP_SERVER_URL not configured"}

    try:
        res = requests.post(
            f"{NLP_URL}/nlp/ask",
            json={
                "question": request.get("message", ""),
                "history": request.get("history", []),
                "use_rag": True,
                "temperature": 0.3,
                "max_tokens": 512
            },
            timeout=60
        )
        res.raise_for_status()
        data = res.json()
        return {
            "answer": data["answer"],
            "sources": data.get("sources", []),
            "elapsed": data.get("elapsed_seconds", 0)
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/api/v1/nlp/health")
async def nlp_health():
    """Health check du serveur NLP"""
    if not NLP_URL:
        return {"status": "error", "message": "NLP_SERVER_URL not set"}

    try:
        res = requests.get(f"{NLP_URL}/nlp/health", timeout=5)
        return res.json()
    except:
        return {"status": "error", "message": "NLP server unreachable"}

@app.get("/")
def root():
    return {
        "status": "ok",
        "modules": ["pdm", "nlp"],
        "nlp_connected": bool(NLP_URL)
    }

In [ ]:
import { useState, useRef, useEffect } from "react";
import axios from "axios";
import { Send, Bot, User } from "lucide-react";

export default function NLPChat() {
  const [query, setQuery] = useState("");
  const [messages, setMessages] = useState([]);
  const [loading, setLoading] = useState(false);

  const bottomRef = useRef(null);

  useEffect(() => {
    bottomRef.current?.scrollIntoView({ behavior: "smooth" });
  }, [messages]);

  const sendMessage = async () => {
    if (!query.trim()) return;

    const userMsg = {
      role: "user",
      text: query,
    };

    setMessages((prev) => [...prev, userMsg]);
    setQuery("");
    setLoading(true);

    try {
      const res = await axios.post("http://localhost:8000/nlp/ask", {
        question: userMsg.text,
      });

      const botMsg = {
        role: "bot",
        text: res.data.answer || "No response available.",
        sources: res.data.sources || [],
      };

      setMessages((prev) => [...prev, botMsg]);
    } catch (err) {
      console.error(err);

      const fallbackMsg = {
        role: "bot",
        text: "The AI assistant is temporarily unavailable.",
        sources: [],
      };

      setMessages((prev) => [...prev, fallbackMsg]);
    } finally {
      setLoading(false);
    }
  };

  return (
    <div
      className="chatPage"
      style={{
        height: "100%",
        width: "100%",
      }}
    >
      {/* HEADER */}
      <div className="chatHeader">
        <div className="title">Engine Copilot</div>
        <div className="subtitle">
          AI-powered maintenance assistant for turbofan diagnostics
        </div>
      </div>

      {/* CHAT */}
      <div className="chatBox">
        {messages.length === 0 && (
          <div className="emptyState">
            <Bot size={28} />
            <div className="emptyTitle">
              Ask questions or summarize maintenance reports
            </div>

            <div className="exampleList">
              <div
                className="exampleCard"
                onClick={() => setQuery("Summarize the maintenance report")}
              >
                Summarize maintenance manuals
              </div>

              <div
                className="exampleCard"
                onClick={() => setQuery("What causes low engine efficiency?")}
              >
                Ask technical questions
              </div>
            </div>
          </div>
        )}

        {messages.map((m, i) => (
          <div key={i} className={`msg ${m.role === "user" ? "user" : "bot"}`}>
            <div className="icon">
              {m.role === "user" ? <User size={14} /> : <Bot size={14} />}
            </div>

            <div className="bubble">
              <div className="text">{m.text}</div>

              {m.sources?.length > 0 && (
                <div className="sources">
                  <div className="sourcesTitle">Sources</div>

                  {m.sources.slice(0, 3).map((s, idx) => (
                    <div key={idx} className="sourceItem">
                      {typeof s === "string" ? s.slice(0, 140) : ""}
                      ...
                    </div>
                  ))}
                </div>
              )}
            </div>
          </div>
        ))}

        {loading && (
          <div className="msg bot">
            <div className="icon">
              <Bot size={14} />
            </div>

            <div className="bubble typing">Processing request...</div>
          </div>
        )}

        <div ref={bottomRef} />
      </div>

      {/* INPUT */}
      <div className="chatInput">
        <input
          value={query}
          onChange={(e) => setQuery(e.target.value)}
          placeholder="Ask about engines, failures, maintenance reports..."
          onKeyDown={(e) => e.key === "Enter" && sendMessage()}
        />

        <button onClick={sendMessage}>
          <Send size={16} />
        </button>
      </div>

      {/* STYLE */}
      <style jsx>{`
        .chatPage {
          height: 100%;
          width: 100%;
          display: flex;
          flex-direction: column;
          background: var(--bg);
          color: var(--text);
          overflow: hidden;
        }

        .chatHeader {
          padding: 16px 20px;
          border-bottom: 1px solid var(--border);
          flex-shrink: 0;
        }

        .title {
          font-size: 18px;
          font-weight: 700;
          color: var(--text);
        }

        .subtitle {
          font-size: 12px;
          color: var(--text2);
          margin-top: 4px;
        }

        .chatBox {
          flex: 1;
          overflow-y: auto;
          padding: 20px;
          display: flex;
          flex-direction: column;
          gap: 14px;
        }

        .emptyState {
          margin: auto;
          display: flex;
          flex-direction: column;
          align-items: center;
          text-align: center;
          gap: 18px;
          color: var(--text2);
          max-width: 520px;
        }

        .emptyTitle {
          font-size: 15px;
          font-weight: 600;
          color: var(--text);
        }

        .exampleList {
          display: grid;
          grid-template-columns: 1fr;
          gap: 10px;
          width: 100%;
        }

        .exampleCard {
          padding: 12px 14px;
          border-radius: 12px;
          background: var(--bg2);
          border: 1px solid var(--border);
          cursor: pointer;
          transition: all 0.2s ease;
          font-size: 13px;
        }

        .exampleCard:hover {
          border-color: rgba(56, 189, 248, 0.3);
          background: rgba(56, 189, 248, 0.05);
        }

        .msg {
          display: flex;
          gap: 10px;
          max-width: 85%;
        }

        .msg.user {
          align-self: flex-end;
          flex-direction: row-reverse;
        }

        .icon {
          width: 30px;
          height: 30px;
          border-radius: 10px;
          background: var(--bg2);
          display: flex;
          align-items: center;
          justify-content: center;
          color: #38bdf8;
          flex-shrink: 0;
        }

        .bubble {
          padding: 12px 14px;
          border-radius: 14px;
          background: var(--bg2);
          border: 1px solid var(--border);
          font-size: 13px;
          line-height: 1.6;
          white-space: pre-wrap;
        }

        .msg.user .bubble {
          background: rgba(56, 189, 248, 0.08);
          border: 1px solid rgba(56, 189, 248, 0.2);
        }

        .sources {
          margin-top: 14px;
          padding-top: 10px;
          border-top: 1px solid rgba(255, 255, 255, 0.05);
        }

        .sourcesTitle {
          font-size: 11px;
          font-weight: 700;
          color: var(--text2);
          margin-bottom: 6px;
          text-transform: uppercase;
          letter-spacing: 0.5px;
        }

        .sourceItem {
          padding: 8px;
          border-radius: 8px;
          background: rgba(255, 255, 255, 0.03);
          margin-bottom: 6px;
          font-size: 11px;
          color: var(--text2);
        }

        .chatInput {
          display: flex;
          gap: 10px;
          padding: 16px;
          border-top: 1px solid var(--border);
          background: var(--bg);
          flex-shrink: 0;
        }

        .chatInput input {
          flex: 1;
          padding: 12px 14px;
          border-radius: 12px;
          border: 1px solid var(--border);
          background: var(--bg2);
          color: var(--text);
          outline: none;
          font-size: 13px;
        }

        .chatInput input:focus {
          border-color: rgba(56, 189, 248, 0.4);
        }

        .chatInput button {
          width: 44px;
          height: 44px;
          border-radius: 12px;
          border: none;
          background: #38bdf8;
          color: #06111f;
          cursor: pointer;
          display: flex;
          align-items: center;
          justify-content: center;
          transition: transform 0.2s ease;
          flex-shrink: 0;
        }

        .chatInput button:hover {
          transform: scale(1.05);
        }

        .typing {
          opacity: 0.7;
          font-style: italic;
        }
      `}</style>
    </div>
  );
}
